PART 1: COMBINED STRESS CALCULATIONS

Interfacial stress pipeline for PCB laminate moisture/thermal analysis

Implements the 5 key equations in Hygro-Thermo-Mechanical Modelling:
1. Fick's second law of diffusion        (governing PDE - solved in plane-sheet analytical mass uptake)
2. Plane-sheet analytical mass uptake    -> moisture_uptake_ratio()
3. Hygroscopic strain                    -> hygroscopic_strain()
4. Thermal mismatch strain                -> thermal_mismatch_strain()
5. Constrained comparative stress index  -> combined_stress_index()

Workflow of this notebook:
- **Part 1** defines the equations and `compute_stress()`.
- **Part 2** loads the material library (input parameters only) from `Formula_sheet_V2.xlsx`.
- **Part 3** runs the pipeline for the material grades you select.
- **Part 4** checks the notebook results against the values calculated in the Excel sheet.

**Source update:** Part 2B now merges the Streamlit `material_presets.json` with the Excel file, using the source priority you choose. Part 5 saves separate Morris and Sobol charts and tables for each selected grade.


In [19]:
# Import libraries
from pathlib import Path
import re

import numpy as np
import pandas as pd
import openpyxl

In [20]:
# Constants
MOLAR_MASS_WATER = 0.018015  # kg/mol, used to convert C_t from mol/m^3 to kg/m^3

In [21]:
# EQUATION 2 — Plane-sheet analytical mass uptake (solves Eq. 1, Fick's 2nd law)
def moisture_uptake_ratio(D, h, t, n_terms=50):
    """
    Mt/M_inf for 1D Fickian diffusion in a plane sheet (Crank's solution).
    D : diffusivity (m^2/s), h : thickness (m), t : time (s)
    """
    n = np.arange(n_terms)
    exponent = -((2 * n + 1) ** 2) * (np.pi ** 2) * D * t / (h ** 2)
    terms = (1.0 / (2 * n + 1) ** 2) * np.exp(exponent)
    return 1.0 - (8.0 / np.pi ** 2) * np.sum(terms)

In [22]:
# EQUATION 3 — Hygroscopic strain
def hygroscopic_strain(beta, W_kg_m3):
    """beta: CHE (m^3/kg), W: moisture concentration (kg/m^3)."""
    return beta * W_kg_m3

In [23]:
# EQUATION 4 — Thermal mismatch strain
def thermal_mismatch_strain(alpha_lam_ppm, alpha_cu_ppm, delta_T):
    """alpha values in 1e-6/K (ppm/K); delta_T in K."""
    alpha_lam = alpha_lam_ppm * 1e-6
    alpha_cu = alpha_cu_ppm * 1e-6
    return (alpha_lam - alpha_cu) * delta_T

In [24]:
# EQUATION 5 — Constrained comparative stress index
def combined_stress_index(E_eff, nu_eff, eps_hygro, eps_thermal):
    """E in MPa, nu dimensionless -> returns stress index in MPa."""
    return (E_eff / (1 - nu_eff)) * (eps_hygro + eps_thermal)

In [25]:
def compute_stress(D, Csat, h, t, beta, alpha_lam, alpha_cu, delta_T, E, nu, n_terms=50):
    """
    Runs all 5 equations for one manually-specified combination of inputs and
    returns the final stress index + intermediate values along the way.

    Inputs:
      D          : diffusivity (m^2/s)
      Csat       : saturation moisture concentration (mol/m^3)
      h          : laminate thickness (m)
      t          : conditioning time (s)
      beta       : hygroscopic swelling coefficient / CHE (m^3/kg)
      alpha_lam  : laminate CTE (ppm/K)
      alpha_cu   : copper CTE (ppm/K)
      delta_T    : temperature change for thermal mismatch (K)
      E          : effective modulus (MPa)
      nu         : Poisson's ratio (dimensionless)
      n_terms    : number of series terms for the moisture uptake sum (default 50)

    Returns a dict with:
      Mt_Minf           -- moisture uptake ratio (Eq. 2)
      C_t_mol_m3        -- moisture concentration at time t, mol/m^3
      W_kg_m3           -- moisture concentration at time t, kg/m^3
      eps_hygro         -- hygroscopic strain (Eq. 3)
      eps_thermal       -- thermal mismatch strain (Eq. 4)
      Stress_Index_MPa  -- combined stress index (Eq. 5)
    """
    mt_minf = moisture_uptake_ratio(D, h, t, n_terms=n_terms)
    C_t_mol_m3 = mt_minf * Csat
    W_kg_m3 = C_t_mol_m3 * MOLAR_MASS_WATER

    eps_h = hygroscopic_strain(beta, W_kg_m3)
    eps_t = thermal_mismatch_strain(alpha_lam, alpha_cu, delta_T)
    stress_index = combined_stress_index(E, nu, eps_h, eps_t)

    return {
        'Mt_Minf': mt_minf,
        'C_t_mol_m3': C_t_mol_m3,
        'W_kg_m3': W_kg_m3,
        'eps_hygro': eps_h,
        'eps_thermal': eps_t,
        'Stress_Index_MPa': stress_index,
    }

PART 2: MATERIAL LIBRARY

Material parameters are read from the Moisture and Thermal sheets of `Formula_sheet_V2.xlsx`. Only the **input** columns are loaded into the library; the calculations themselves are done in this notebook. Every material in the library comes from the formula sheet, so to add or change a material, edit the sheet and re-run from here.

In [26]:
# PART 2 — File paths. Put the JSON + Excel + notebook in the same folder.
EXCEL_PATH = Path('Formula_sheet_V2(4).xlsx')
if not EXCEL_PATH.is_file():
    EXCEL_PATH = Path('Formula_sheet_V2.xlsx')
if not EXCEL_PATH.is_file():
    raise FileNotFoundError('Excel material file not found. Place Formula_sheet_V2(4).xlsx next to notebook.')
JSON_PATH = Path('material_presets.json')
if not JSON_PATH.is_file():
    raise FileNotFoundError('Streamlit material_presets.json not found next to the notebook.')
# If a grade/condition exists in both sources, choose which properties take precedence.
PRESET_PRIORITY = 'json'  # 'json' = Streamlit wins; 'excel' = Excel wins
MOISTURE_SHEET = 'Moisture'
THERMAL_SHEET = 'Thermal'
HEADER_ROW = 4
print('Excel:', EXCEL_PATH.resolve())
print('Streamlit JSON:', JSON_PATH.resolve())


Excel: C:\Users\yong jiang\Desktop\Y4S1\3015\Updated 111026\Formula_sheet_V2.xlsx
Streamlit JSON: C:\Users\yong jiang\Desktop\Y4S1\3015\Updated 111026\material_presets.json


In [27]:
# Columns are found by the start of their header text in row 4, so the reader
# still works if columns in the sheet are moved or inserted.

# Descriptive columns (Moisture sheet)
INFO_COLS = {"supplier": "Supplier", "grade": "Grade", "class": "Class", "condition": "Condition"}

# Input parameters for compute_stress(): key = argument name
MOISTURE_INPUTS = {"D": "D (m2/s)", "Csat": "Csat", "h": "h (m)", "t": "t (s)"}
THERMAL_INPUTS = {"beta": "β", "alpha_lam": "αlam", "alpha_cu": "αCu",
                  "delta_T": "ΔT", "E": "Eeff", "nu": "νeff"}
INPUT_NAMES = list(MOISTURE_INPUTS) + list(THERMAL_INPUTS)

# Excel output columns - only used for the validation in Part 4
MOISTURE_OUTPUTS = {"Mt_Minf": "Mt/M∞ (fractional", "W_kg_m3": "W ="}
THERMAL_OUTPUTS = {"eps_hygro": "εh", "eps_thermal": "εT", "Stress_Index_MPa": "σ_index"}


def _find_columns(ws, wanted):
    """Return {key: column_index} by matching header text in HEADER_ROW."""
    headers = {c.column: str(c.value).strip() for c in ws[HEADER_ROW] if c.value is not None}
    found = {}
    for key, prefix in wanted.items():
        matches = [col for col, text in headers.items() if text.startswith(prefix)]
        if not matches:
            raise KeyError(f"Sheet '{ws.title}': no header starting with '{prefix}'")
        found[key] = matches[0]
    return found


_NUMERIC_EXPR = re.compile(r"^[0-9eE.+\-*/^() ]+$")

def _cell_value(ws_vals, ws_form, row, col):
    """
    Value of a cell. Uses the value Excel last calculated; if that is missing (file
    never recalculated), evaluates simple input formulas such as '=1.15*10^-13'
    or same-sheet references such as '=$C$20'.
    """
    v = ws_vals.cell(row, col).value
    if v is not None:
        return v
    f = ws_form.cell(row, col).value
    if not (isinstance(f, str) and f.startswith("=")):
        return f
    expr = f[1:].strip()
    ref = re.fullmatch(r"\$?([A-Z]{1,3})\$?(\d+)", expr)
    if ref:
        c = openpyxl.utils.column_index_from_string(ref.group(1))
        return _cell_value(ws_vals, ws_form, int(ref.group(2)), c)
    if _NUMERIC_EXPR.match(expr):
        return eval(expr.replace("^", "**"), {"__builtins__": {}})
    return None


def load_formula_sheet(path=EXCEL_PATH):
    """
    Returns two DataFrames with the same index:
      library        - descriptive columns + input parameters for compute_stress()
      excel_outputs  - the values the Excel sheet calculated (for validation only)
    Rows without a complete set of input parameters are left out of both.
    """
    wb_v = openpyxl.load_workbook(path, data_only=True)   # cached values
    wb_f = openpyxl.load_workbook(path, data_only=False)  # formulas (fallback only)
    mv, mf = wb_v[MOISTURE_SHEET], wb_f[MOISTURE_SHEET]
    tv, tf = wb_v[THERMAL_SHEET], wb_f[THERMAL_SHEET]

    m_cols = _find_columns(mf, {**INFO_COLS, **MOISTURE_INPUTS, **MOISTURE_OUTPUTS})
    t_cols = _find_columns(tf, {"grade": "Grade", "condition": "Condition",
                                **THERMAL_INPUTS, **THERMAL_OUTPUTS})

    lib_rows, out_rows, skipped = [], [], []
    for r in range(HEADER_ROW + 1, mf.max_row + 1):
        m = {k: _cell_value(mv, mf, r, c) for k, c in m_cols.items()}
        if m["grade"] is None:
            continue
        t = {k: _cell_value(tv, tf, r, c) for k, c in t_cols.items()}
        # The Thermal sheet uses the same row order; make sure it is the same material
        if (t["grade"], t["condition"]) != (m["grade"], m["condition"]):
            t = {k: None for k in t}

        params = {k: pd.to_numeric((m | t)[k], errors="coerce") for k in INPUT_NAMES}
        if any(pd.isna(v) for v in params.values()):
            skipped.append(f"row {r}: {m['grade']} {m['condition']}")
            continue
        lib_rows.append({"excel_row": r, **{k: m[k] for k in INFO_COLS}, **params})
        out_rows.append({k: pd.to_numeric((m | t)[k], errors="coerce")
                         for k in list(MOISTURE_OUTPUTS) + list(THERMAL_OUTPUTS)})

    if skipped:
        print("Skipped (incomplete inputs in sheet):", *skipped, sep="\n  ")
    return pd.DataFrame(lib_rows), pd.DataFrame(out_rows)


library, excel_outputs = load_formula_sheet()
print(f"Material library: {len(library)} grade/condition rows from {EXCEL_PATH.name}")

Material library: 52 grade/condition rows from Formula_sheet_V2.xlsx


In [28]:
# PART 2B — Merge the Streamlit JSON presets with the Excel material library.
# Identity is (supplier, grade, temperature/RH condition), NOT row number.
# Default: JSON has priority for matching entries, Excel supplies any grades missing from JSON.
# This also makes the JSON-only package substrate grades selectable.
import json
from collections import Counter

PROP_MAP = {
    'E':'E', 'nu':'nu', 'alpha_lam':'alpha_lam', 'alpha_cu':'alpha_cu',
    'D':'D', 'Csat':'Csat', 'beta':'beta',
    'h_mm':'h', 't_hours':'t', 'delta_T':'delta_T'
}
def preset_identity(supplier, grade, condition):
    def tidy(x): return ' '.join(str(x).casefold().split())
    return tidy(supplier), tidy(grade), tidy(condition)

def load_streamlit_presets(path):
    with open(path, 'r', encoding='utf-8') as fh:
        doc = json.load(fh)
    if not isinstance(doc.get('presets'), list):
        raise ValueError('Streamlit JSON must contain a list named presets.')
    rows = []
    seen = set()
    for p in doc['presets']:
        key = preset_identity(p['supplier'], p['grade'], p['condition'])
        if key in seen:
            raise ValueError(f'Duplicate Streamlit JSON preset: {key}')
        seen.add(key)
        vals = p['properties']
        missing = set(PROP_MAP) - set(vals)
        if missing:
            raise ValueError(f'{p["name"]}: missing properties: {sorted(missing)}')
        row = {k:float(vals[json_key]) for json_key,k in PROP_MAP.items()}
        row['h'] = float(vals['h_mm']) / 1000.0  # mm -> m
        row['t'] = float(vals['t_hours']) * 3600.0 # hours -> seconds
        rows.append({'supplier':p['supplier'], 'grade':p['grade'],
            'condition':p['condition'], 'class':p['material_class'],
            'preset_name':p['name'], 'preset_key':key, **row})
    return rows

json_presets = load_streamlit_presets(JSON_PATH)
json_by_id = {r['preset_key']:r for r in json_presets}
if PRESET_PRIORITY not in ('json','excel'):
    raise ValueError('PRESET_PRIORITY must be json or excel')

# Retain ALL Excel materials; replace only matched entries when JSON has priority.
library = library.copy()
library['preset_source'] = 'Excel only'
library['preset_name'] = (library['supplier'].astype(str) + ' ' +
                          library['grade'].astype(str) + ' — ' + library['condition'].astype(str))
library['material'] = library['supplier'].astype(str) + ' ' + library['grade'].astype(str)
audit_rows=[]
excel_keys=set()
for ix,row in library.iterrows():
    key=preset_identity(row['supplier'],row['grade'],row['condition'])
    if key in excel_keys:
        raise ValueError('Duplicate Excel preset identity: '+str(key))
    excel_keys.add(key)
    match=json_by_id.get(key)
    if match is None:
        audit_rows.append(dict(index=ix,material=row['material'],condition=row['condition'],
                               property='(row)',excel='',json='',source='Excel only',different=False))
        continue
    for p in INPUT_NAMES:
        excel_value, json_value=float(row[p]),float(match[p])
        changed=not np.isclose(excel_value,json_value,rtol=1e-8,atol=1e-16)
        audit_rows.append(dict(index=ix,material=row['material'],condition=row['condition'],
            property=p,excel=excel_value,json=json_value,
            source='JSON' if PRESET_PRIORITY=='json' else 'Excel',different=changed))
        if PRESET_PRIORITY=='json': library.at[ix,p]=json_value
    library.at[ix,'preset_source'] = ('Streamlit JSON (matched Excel)' if PRESET_PRIORITY=='json'
                                  else 'Excel (matched Streamlit JSON)')
    library.at[ix,'preset_name'] = match['preset_name']

# Include JSON-only grades. Their Excel comparison is intentionally unavailable.
for key,p in json_by_id.items():
    if key in excel_keys:continue
    ix=int(library.index.max())+1
    new_row={'excel_row':np.nan,'supplier':p['supplier'],'grade':p['grade'],
             'class':p['class'],'condition':p['condition'],
             'material':str(p['supplier'])+' '+str(p['grade']),
             'preset_name':p['preset_name'], 'preset_source':'Streamlit JSON only',
             **{key:p[key] for key in INPUT_NAMES}}
    library.loc[ix] = new_row
    audit_rows.append(dict(index=ix,material=new_row['material'],condition=p['condition'],
                           property='(row)',excel='',json='',source='Streamlit JSON only',different=False))

# Keep cached Excel outputs in sync with the new index (JSON-only rows have no Excel outputs).
excel_outputs=excel_outputs.reindex(library.index)
preset_audit=pd.DataFrame(audit_rows)
preset_audit.to_csv('Material_source_audit.csv',index=False)
print('Presets now available:',len(library),'rows from BOTH files')
print('Material sources:',dict(Counter(library['preset_source'])))
print('Different matching Excel vs JSON property values:',int(preset_audit['different'].sum()))
if preset_audit['different'].any():
    display(preset_audit.loc[preset_audit['different']].head(10))
# View this table when choosing SELECTED indices or SELECTED_PRESET_NAMES.
display(library[['supplier','grade','class','condition','preset_source','preset_name']].head(12))


Presets now available: 52 rows from BOTH files
Material sources: {'Excel only': 32, 'Streamlit JSON (matched Excel)': 20}
Different matching Excel vs JSON property values: 0


,supplier,grade,class,condition,preset_source,preset_name
0,Panasonic,CV8710U,EMC encapsulation,85°C/85%RH,Excel only,Panasonic CV8710U — 85°C/85%RH
1,Panasonic,CV8710U,EMC encapsulation,60°C/60%RH,Excel only,Panasonic CV8710U — 60°C/60%RH
2,Shengyi,SI10US/SI13U,Package substrate laminate/core,85°C/85%RH,Streamlit JSON (matched Excel),Shengyi SI10US/SI13U — 85°C/85%RH
3,Shengyi,SI10US/SI13U,Package substrate laminate/core,60°C/60%RH,Streamlit JSON (matched Excel),Shengyi SI10US/SI13U — 60°C/60%RH
4,Ajinomoto,ABF GX-T31,ABF build-up dielectric,85°C/85%RH,Excel only,Ajinomoto ABF GX-T31 — 85°C/85%RH
5,Ajinomoto,ABF GX-T31,ABF build-up dielectric,60°C/60%RH,Excel only,Ajinomoto ABF GX-T31 — 60°C/60%RH
6,Sumitomo Bakelite,EME-A700,EMC encapsulation,85°C/85%RH,Excel only,Sumitomo Bakelite EME-A700 — 85°C/85%RH
7,Sumitomo Bakelite,EME-A700,EMC encapsulation,60°C/60%RH,Excel only,Sumitomo Bakelite EME-A700 — 60°C/60%RH
8,Resonac,CEL-9200,EMC encapsulation,85°C/85%RH,Excel only,Resonac CEL-9200 — 85°C/85%RH
9,Resonac,CEL-9200,EMC encapsulation,60°C/60%RH,Excel only,Resonac CEL-9200 — 60°C/60%RH


In [29]:
# PART 2C — Browse the merged presets (Excel + Streamlit).
pd.set_option('display.max_rows', 200)
display(library[['supplier','grade','class','condition','preset_source','preset_name']])


,supplier,grade,class,condition,preset_source,preset_name
0,Panasonic,CV8710U,EMC encapsulation,85°C/85%RH,Excel only,Panasonic CV8710U — 85°C/85%RH
1,Panasonic,CV8710U,EMC encapsulation,60°C/60%RH,Excel only,Panasonic CV8710U — 60°C/60%RH
2,Shengyi,SI10US/SI13U,Package substrate laminate/core,85°C/85%RH,Streamlit JSON (matched Excel),Shengyi SI10US/SI13U — 85°C/85%RH
3,Shengyi,SI10US/SI13U,Package substrate laminate/core,60°C/60%RH,Streamlit JSON (matched Excel),Shengyi SI10US/SI13U — 60°C/60%RH
4,Ajinomoto,ABF GX-T31,ABF build-up dielectric,85°C/85%RH,Excel only,Ajinomoto ABF GX-T31 — 85°C/85%RH
5,Ajinomoto,ABF GX-T31,ABF build-up dielectric,60°C/60%RH,Excel only,Ajinomoto ABF GX-T31 — 60°C/60%RH
6,Sumitomo Bakelite,EME-A700,EMC encapsulation,85°C/85%RH,Excel only,Sumitomo Bakelite EME-A700 — 85°C/85%RH
7,Sumitomo Bakelite,EME-A700,EMC encapsulation,60°C/60%RH,Excel only,Sumitomo Bakelite EME-A700 — 60°C/60%RH
8,Resonac,CEL-9200,EMC encapsulation,85°C/85%RH,Excel only,Resonac CEL-9200 — 85°C/85%RH
9,Resonac,CEL-9200,EMC encapsulation,60°C/60%RH,Excel only,Resonac CEL-9200 — 60°C/60%RH


PART 3: RUN THE STRESS CALCULATION PIPELINE

Select materials by index from the library above. Parameters in `OVERRIDES` replace the sheet values for every selected material, which lets you change test conditions (thickness, time, ΔT, etc.) from the notebook without editing the Excel file.

In [30]:
# PART 3 — Standard calculator selection by MERGED library index.
# For Morris/Sobol by Streamlit preset NAME, use SELECTED_PRESET_NAMES in Part 5A instead.
SELECTED = [0,1]  # indices from the displayed merged library (not Excel worksheet numbers)
OVERRIDES = {}  # e.g. {'t': 7*86400}; affects Part 3 calculator only, not Part 5


In [31]:
def extract_parameters(idx, overrides=None, lib=library):
    """Return the compute_stress() inputs for the material at index idx."""
    params = {k: float(lib.loc[idx, k]) for k in INPUT_NAMES}
    unknown = set(overrides or {}) - set(INPUT_NAMES)
    if unknown:
        raise KeyError(f"Unknown override(s) {unknown}; valid names: {INPUT_NAMES}")
    params.update(overrides or {})
    return params


def run_selected(indices, overrides=None, lib=library):
    """Run compute_stress() for each selected index; return inputs + results."""
    rows = []
    for idx in indices:
        params = extract_parameters(idx, overrides, lib)
        rows.append({
            "index": idx,
            "material": f"{lib.loc[idx, 'supplier']} {lib.loc[idx, 'grade']}",
            "condition": lib.loc[idx, "condition"],
            **params,
            **compute_stress(**params),
        })
    return pd.DataFrame(rows).set_index("index")


results = run_selected(SELECTED, OVERRIDES)

# Input parameters used
results[["material", "condition", *INPUT_NAMES]]

,material,condition,D,Csat,h,t,beta,alpha_lam,alpha_cu,delta_T,E,nu
index,,,,,,,,,,,,
0,Panasonic CV8710U,85°C/85%RH,1.150000e-13,190.5,0.0016,86400.0,0.00018,9.0,16.5,60.0,24000.0,0.25
1,Panasonic CV8710U,60°C/60%RH,3.200000e-14,175.0,0.0016,86400.0,0.00018,9.0,16.5,35.0,24000.0,0.25


In [32]:
# Calculation results
results[["material", "condition", "Mt_Minf", "C_t_mol_m3", "W_kg_m3",
         "eps_hygro", "eps_thermal", "Stress_Index_MPa"]]

,material,condition,Mt_Minf,C_t_mol_m3,W_kg_m3,eps_hygro,eps_thermal,Stress_Index_MPa
index,,,,,,,,
0,Panasonic CV8710U,85°C/85%RH,0.140595,26.783408,0.482503,0.000087,-0.000450,-11.620782
1,Panasonic CV8710U,60°C/60%RH,0.074165,12.978813,0.233813,0.000042,-0.000262,-7.053235


PART 4: VALIDATION AGAINST THE EXCEL FORMULA SHEET

Compares the notebook results with the values the Excel sheet calculated for the same rows. The check is skipped if `OVERRIDES` is in use, since the Excel values were calculated with the sheet's own inputs.

In [33]:
REL_TOL = 1e-6
CHECK_LABELS = {'Mt_Minf':'Mt/M∞', 'W_kg_m3':'W (kg/m³)', 'eps_hygro':'εh',
                'eps_thermal':'εT','Stress_Index_MPa':'σ_index (MPa)'}
def compare_with_excel(results, xl=excel_outputs, rel_tol=REL_TOL):
    rows=[]
    for idx,r in results.iterrows():
        for col,label in CHECK_LABELS.items():
            source=str(library.loc[idx,'preset_source'])
            nb_val=r[col]; xl_val=xl.loc[idx,col] if idx in xl.index else np.nan
            if source.startswith('Streamlit'):
                rel=np.nan
                status='JSON source: Excel reference not applicable'
            elif pd.isna(xl_val):
                rel=np.nan;status='No Excel cached result'
            else:
                rel=abs(nb_val-xl_val)/(abs(xl_val) if xl_val!=0 else 1.0)
                status='PASS' if rel<=rel_tol else 'CHECK'
            rows.append({'index':idx,'material':r['material'],'condition':r['condition'],
                 'source':source,'quantity':label,'notebook':nb_val,'excel':xl_val,
                 'rel_diff':rel,'status':status})
    return pd.DataFrame(rows)
if OVERRIDES:
    print('Validation skipped because Part 3 OVERRIDES is active.')
else:
    comparison=compare_with_excel(results)
    print('Excel comparison CHECK count:',int((comparison['status']=='CHECK').sum()))
    display(comparison)


Excel comparison CHECK count: 0


,index,material,condition,source,quantity,notebook,excel,rel_diff,status
0,0,Panasonic CV8710U,85°C/85%RH,Excel only,Mt/M∞,0.140595,0.140595,3.948293e-16,PASS
1,0,Panasonic CV8710U,85°C/85%RH,Excel only,W (kg/m³),0.482503,0.482503,4.601931e-16,PASS
2,0,Panasonic CV8710U,85°C/85%RH,Excel only,εh,0.000087,0.000087,1.404398e-15,PASS
3,0,Panasonic CV8710U,85°C/85%RH,Excel only,εT,-0.000450,-0.000450,2.409338e-16,PASS
4,0,Panasonic CV8710U,85°C/85%RH,Excel only,σ_index (MPa),-11.620782,-11.620782,1.528604e-16,PASS
5,1,Panasonic CV8710U,60°C/60%RH,Excel only,Mt/M∞,0.074165,0.074165,9.356067e-16,PASS
6,1,Panasonic CV8710U,60°C/60%RH,Excel only,W (kg/m³),0.233813,0.233813,1.187083e-16,PASS
7,1,Panasonic CV8710U,60°C/60%RH,Excel only,εh,0.000042,0.000042,0.000000e+00,PASS
8,1,Panasonic CV8710U,60°C/60%RH,Excel only,εT,-0.000262,-0.000262,2.065147e-16,PASS
9,1,Panasonic CV8710U,60°C/60%RH,Excel only,σ_index (MPa),-7.053235,-7.053235,7.555498e-16,PASS


Notes on the validation:
- Excel uses the 4-term series when Fo ≥ 0.05 and the short-time solution when Fo < 0.05. The notebook uses a 50-term series throughout, which matches both well within tolerance at the Fo values in the sheet. Below Fo ≈ 1e-5, 50 terms are no longer enough.
- The Excel values are those last calculated when the workbook was saved. After editing inputs, save the workbook in Excel before re-running. A file saved by a script without recalculating will show "no Excel value".

# PART 5 — INDEPENDENT MORRIS AND SOBOL REPORTS FOR EACH SOURCE PRESET

This edition merges the **Streamlit `material_presets.json`** with the **Excel `Formula_sheet_V2(4).xlsx`** library. Each preset is matched using **supplier + grade + condition**.

**Source precedence:** Set `PRESET_PRIORITY = 'json'` in Part 2. Matching materials take their model inputs from Streamlit. Excel fills in grades not contained in JSON (including EMC). Set the priority to `'excel'` to prefer Excel. `Material_source_audit.csv` records source differences instead of hiding them.

**Selecting one preset:** Set `SENSITIVITY_SCOPE = 'selected'` and write its *exact Streamlit name* into `SELECTED_PRESET_NAMES`, e.g. `['Isola 370HR — 85°C/85%RH']`. If this list is empty, the notebook uses indices from `SELECTED` in Part 3. The selected condition alone gets the intrinsic analysis; external sensitivity needs both 60/60 and 85/85 endpoints.

**Selecting all:** Set `SENSITIVITY_SCOPE = 'all'`, generating an individual graph and separate tables for each material and each applicable condition. Standard families are PCB laminate and EMC. Set `INCLUDE_PACKAGE_SUBSTRATE = True` to include the Streamlit package-substrate core presets as well. ABF is not included.

**Physical assumptions:** Intrinsic D: nominal÷2 to nominal×2 (log-uniform); Csat, CME, CTE, E: ±10% (uniform). ν, thickness, copper CTE and 24-hour exposure remain fixed from the chosen condition's preset. External study varies combined T/RH severity and log-uniform time 0.25–7 days, with matched 60/60 and 85/85 endpoints. Input ranges are *illustrative uncertainty assumptions*, not measured tolerances. The stress is a simplified biaxial constraint INDEX, not a 3D copper-pad field.


In [34]:
# PART 5A — Analysis settings: edit these fields only.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import qmc
from pathlib import Path
import re

SENSITIVITY_SCOPE = 'all'           # 'all' or 'selected'
SELECTED_PRESET_NAMES = []          # e.g. ['Isola 370HR — 85°C/85%RH']
INCLUDE_PACKAGE_SUBSTRATE = True    # Also analyse Streamlit's substrate laminate/core grades
OUTDIR = Path('Dual_source_individual_reports')
OUTDIR.mkdir(parents=True,exist_ok=True)
FAMILIES = ['PCB laminate','EMC encapsulation']
if INCLUDE_PACKAGE_SUBSTRATE:
    FAMILIES.append('Package substrate laminate/core')
LOW_CONDITION='60°C/60%RH'
HIGH_CONDITION='85°C/85%RH'
TIME_DAYS=(0.25,7.0)
MORRIS_R=40
SOBOL_N=1024
SEED=2026
INTRINSIC_PARAMS=['D','Csat','beta','alpha_lam','E']
INTRINSIC_RANGES={'D':2.0,'Csat':0.10,'beta':0.10,'alpha_lam':0.10,'E':0.10}
OUTPUT_KEYS=['Mt_Minf','W_kg_m3','eps_hygro','Stress_Index_MPa']
OUTPUT_PRETTY={'Mt_Minf':'Fractional uptake Mt/M∞','W_kg_m3':'Absorbed water W (kg/m³)',
    'eps_hygro':'Hygroscopic strain εh','Stress_Index_MPa':'Stress index (MPa)'}
PARAM_PRETTY={'D':'D','Csat':'Csat','beta':'CME β','alpha_lam':'CTE α',
    'E':'Eeff','Coupled_T_RH':'Joint T/RH','Exposure_time':'Time'}
PLOT_INLINE_LIMIT=0

if SENSITIVITY_SCOPE not in ('all','selected'):
    raise ValueError('SENSITIVITY_SCOPE must be all or selected')
if SENSITIVITY_SCOPE=='selected' and not (SELECTED_PRESET_NAMES or SELECTED):
    raise ValueError('Choose one preset by SELECTED_PRESET_NAMES or SELECTED index.')
print('Scope:',SENSITIVITY_SCOPE,'| Streamlit JSON:',JSON_PATH.name,'| Excel:',EXCEL_PATH.name)


Scope: all | Streamlit JSON: material_presets.json | Excel: Formula_sheet_V2.xlsx


In [35]:
# PART 5B — Match both environmental presets for ONE grade (never mix grades).
subset=library.loc[library['class'].isin(FAMILIES)].copy()
subset['material']=subset['supplier'].astype(str)+' '+subset['grade'].astype(str)
pairs=[]
for (family,material),part in subset.groupby(['class','material'],sort=True):
    by_condition=part.set_index('condition')
    if LOW_CONDITION not in by_condition.index or HIGH_CONDITION not in by_condition.index:
        print('SKIP incomplete environment pair:',material)
        continue
    low,high=by_condition.loc[LOW_CONDITION],by_condition.loc[HIGH_CONDITION]
    for k in ['h','beta','alpha_lam','alpha_cu','E','nu']:
        if not np.isclose(float(low[k]),float(high[k]),rtol=1e-8,atol=1e-14):
            raise ValueError(f'{material}: {k} differs across condition presets. Review source audit.')
    if min(low['D'],high['D'],low['Csat'],high['Csat'])<=0:
        raise ValueError(f'{material}: diffusivity and Csat must be positive.')
    pairs.append(dict(family=family,material=material,low=low,high=high,
                      selected_conditions={LOW_CONDITION,HIGH_CONDITION}))

if SENSITIVITY_SCOPE=='selected':
    if SELECTED_PRESET_NAMES:
        # Exact Streamlit preset names OR the merged library's equivalent name.
        chosen=subset.loc[subset['preset_name'].isin(SELECTED_PRESET_NAMES)].copy()
        unknown=set(SELECTED_PRESET_NAMES)-set(chosen['preset_name'])
        if unknown:raise KeyError(f'Preset name(s) not found: {sorted(unknown)}')
    else:
        missing=set(SELECTED)-set(library.index)
        if missing:raise KeyError(f'Unknown selected indices: {missing}')
        chosen=library.loc[list(dict.fromkeys(SELECTED))]
        chosen=chosen.loc[chosen['class'].isin(FAMILIES)].copy()
    chosen['material']=chosen['supplier'].astype(str)+' '+chosen['grade'].astype(str)
    wanted={(r['class'],r['material']) for _,r in chosen.iterrows()}
    pairs=[pair for pair in pairs if (pair['family'],pair['material']) in wanted]
    for pair in pairs:
        pair['selected_conditions']=set(chosen.loc[
            chosen['class'].eq(pair['family']) & chosen['material'].eq(pair['material']),
            'condition'])
    if not pairs:
        raise ValueError('No complete 60/60 and 85/85 pair among selected presets.')

print('Separate material reports:',len(pairs))
for p in pairs:
    print(p['material'],'| source 60/60:',p['low']['preset_source'],
          '| source 85/85:',p['high']['preset_source'],
          '| selected conditions:',sorted(p['selected_conditions']))


Separate material reports: 21
Nitto Denko GE-100 | source 60/60: Excel only | source 85/85: Excel only | selected conditions: ['60°C/60%RH', '85°C/85%RH']
Panasonic CV-8300 | source 60/60: Excel only | source 85/85: Excel only | selected conditions: ['60°C/60%RH', '85°C/85%RH']
Panasonic CV8200 | source 60/60: Excel only | source 85/85: Excel only | selected conditions: ['60°C/60%RH', '85°C/85%RH']
Panasonic CV8700 | source 60/60: Excel only | source 85/85: Excel only | selected conditions: ['60°C/60%RH', '85°C/85%RH']
Panasonic CV8710U | source 60/60: Excel only | source 85/85: Excel only | selected conditions: ['60°C/60%RH', '85°C/85%RH']
Resonac CEL-9200 | source 60/60: Excel only | source 85/85: Excel only | selected conditions: ['60°C/60%RH', '85°C/85%RH']
Resonac CEL-9750 | source 60/60: Excel only | source 85/85: Excel only | selected conditions: ['60°C/60%RH', '85°C/85%RH']
Sumitomo Bakelite EME-A700 | source 60/60: Excel only | source 85/85: Excel only | selected conditions: [

In [36]:
# PART 5C — VECTORISED ORIGINAL FICK + STRAIN + STRESS-INDEX MODEL
# For small Fourier number, use the short-time sheet approximation.
# This avoids truncation error when the diffusion time is very short.
def uptake_sheet(D, h, t):
    D, t = np.broadcast_arrays(np.asarray(D, dtype=float), np.asarray(t, dtype=float))
    Fo = D * t / (float(h) ** 2)                # Dimensionless Fourier number
    answer = np.empty_like(Fo)                 # One moisture-uptake value per sample
    short = Fo < 0.05                          # Early-time Fickian region
    answer[short] = 4.0 * np.sqrt(Fo[short] / np.pi)
    if (~short).any():                         # Use exponential series for later times
        odd = 2 * np.arange(80) + 1            # 80 odd-number terms, 1,3,5,...
        exponent = -np.pi**2 * Fo[~short, None] * odd[None, :]**2
        answer[~short] = 1 - (8/np.pi**2) * np.sum(
            np.exp(exponent) / odd[None, :]**2, axis=1)
    return np.clip(answer, 0.0, 1.0)           # Physical fractional uptake bounds

def physics_response(D, Csat, beta, alpha_lam, E, nu, alpha_cu, delta_T, h, time_s):
    uptake = uptake_sheet(D, h, time_s)        # Mt/M∞ for the 1D plane sheet
    W = uptake * Csat * MOLAR_MASS_WATER       # Moisture mass concentration kg/m³
    eps_h = beta * W                          # Free hygroscopic strain
    eps_t = (alpha_lam - alpha_cu) * 1e-6 * delta_T # Thermal expansion mismatch
    stress = E / (1 - nu) * (eps_h + eps_t)   # Simplified constrained stress INDEX, MPa
    return np.column_stack(np.broadcast_arrays(uptake, W, eps_h, stress))

# Compare the vectorized equation against the existing function at spreadsheet conditions.
for grade in pairs[:3]:
    row = grade["high"]
    expected = compute_stress(**{key:float(row[key]) for key in INPUT_NAMES})
    result = physics_response(row["D"],row["Csat"],row["beta"],row["alpha_lam"],
                              row["E"],row["nu"],row["alpha_cu"],row["delta_T"],
                              row["h"],row["t"])[0]
    assert np.allclose(result, [expected[k] for k in OUTPUT_KEYS], rtol=2e-3, atol=1e-10)
print("Validated vectorized implementation against original calculator on three grades")


Validated vectorized implementation against original calculator on three grades


In [37]:
# PART 5D — TWO INDEPENDENT EXTERNAL SCENARIO INPUTS
# u[:,0] = combined (T,RH) severity: 0=60°C/60%RH; 1=85°C/85%RH.
# u[:,1] = scaled exposure time, log-uniform between TIME_DAYS endpoints.
def evaluate_external(u, pair):
    u = np.atleast_2d(np.asarray(u, dtype=float)) # Rows of normalized sensitivity samples
    s = u[:, 0]                                  # Environment along the coupled T-RH path
    low, high = pair["low"], pair["high"]       # This grade's two source observations
    D = np.exp((1-s)*np.log(low["D"]) + s*np.log(high["D"])) # ASSUMED log interpolation
    Csat = low["Csat"] + s*(high["Csat"]-low["Csat"])         # ASSUMED linear interpolation
    dT = low["delta_T"] + s*(high["delta_T"]-low["delta_T"]) # Temperature change vs reference
    days = TIME_DAYS[0] * (TIME_DAYS[1]/TIME_DAYS[0])**u[:,1] # ASSUMED log-time sampling
    return physics_response(D, Csat, high["beta"], high["alpha_lam"],
                            high["E"], high["nu"], high["alpha_cu"], dT,
                            high["h"], days*86400)

# Sanity check: environmental severity endpoints reproduce the observed source properties.
first = pairs[0]
for severity, point in [(0,first["low"]),(1,first["high"])]:
    x = np.array([[severity, np.log(float(point["t"])/86400/TIME_DAYS[0]) /
                   np.log(TIME_DAYS[1]/TIME_DAYS[0])]])
    prediction = evaluate_external(x,first)[0]
    base = compute_stress(**{k:float(point[k]) for k in INPUT_NAMES})
    assert np.allclose(prediction,[base[k] for k in OUTPUT_KEYS],rtol=2e-3,atol=1e-10)
print("Environmental endpoints verified for:", first["material"])


Environmental endpoints verified for: Nitto Denko GE-100


In [38]:
# PART 5E — INTRINSIC STUDY: ONE GRADE AT ONE FIXED ENVIRONMENT
# This answers a different question: what if the grade's reported properties are uncertain?
# The ranges are HYPOTHETICAL until experimental/datasheet uncertainty is available.
def intrinsic_bounds(row):
    bounds = {}                                # Lower and upper bounds for this particular grade
    for key in INTRINSIC_PARAMS:
        nominal = float(row[key])              # Nominal value from the workbook
        width = INTRINSIC_RANGES[key]          # Assumed uncertainty parameter
        lo, hi = (nominal/width, nominal*width) if key == "D" else (
                  nominal*(1-width), nominal*(1+width))
        if not np.isfinite(lo+hi) or lo >= hi or (key in ("D","Csat","E") and lo<=0):
            raise ValueError(f"Invalid assumed intrinsic bounds: {key} for {row['grade']}")
        bounds[key] = (lo, hi)                  # Each grade uses its OWN nominal properties
    return bounds

def evaluate_intrinsic(u, row):
    u = np.atleast_2d(np.asarray(u,dtype=float)) # Independent unit-square inputs
    bounds = intrinsic_bounds(row)
    varied = {}                                 # Map each column onto property units
    for j,key in enumerate(INTRINSIC_PARAMS):
        lo,hi = bounds[key]
        varied[key] = lo*(hi/lo)**u[:,j] if key=="D" else lo+(hi-lo)*u[:,j]
    return physics_response(varied["D"],varied["Csat"],varied["beta"],
                            varied["alpha_lam"],varied["E"],float(row["nu"]),
                            float(row["alpha_cu"]),float(row["delta_T"]),
                            float(row["h"]),float(row["t"]))


In [39]:
# PART 5F — MORRIS: elementary effects (one-at-a-time moves)
def morris(model, k, r=MORRIS_R, levels=6, seed=SEED):
    rng = np.random.default_rng(seed)             # Reproducible designs
    delta = levels / (2*(levels-1))               # Morris step = 0.6 at 6 levels
    effects = np.zeros((r,k,len(OUTPUT_KEYS)))    # Trajectory × parameter × output
    for i in range(r):
        low_grid = rng.choice(np.arange(levels//2)/(levels-1),size=k)
        sign = rng.choice([-1,1],size=k)          # Random increase or decrease
        x = np.where(sign<0,low_grid+delta,low_grid) # Stay in [0,1] for ± moves
        order = rng.permutation(k)                # Randomized one-at-a-time ordering
        path = [x.copy()]                         # Save initial point
        for j in order:
            x=x.copy(); x[j]+=sign[j]*delta; path.append(x)
        y = model(np.vstack(path))                # Evaluate all (k+1) trajectory points
        for step,j in enumerate(order):
            effects[i,j,:]=(y[step+1]-y[step])/(sign[j]*delta)
    mu_star = np.mean(np.abs(effects),axis=0)     # Mean ABSOLUTE influence in output units
    sigma = np.std(effects,axis=0,ddof=1)        # Nonlinear / interaction variability
    return mu_star,sigma


In [40]:
# PART 5G — SOBOL: Jansen first-order and total-order indices
# Input factors must be treated as INDEPENDENT under the stipulated sample distributions.
def sobol_jansen(model, k, n=SOBOL_N, seed=SEED):
    if n<2 or (n & (n-1)): raise ValueError("SOBOL_N must be a power of two")
    sampler=qmc.Sobol(d=2*k,scramble=True,seed=seed)   # Independent U(0,1) marginals
    sample=sampler.random_base2(m=int(np.log2(n)))     # 2k-dimensional Sobol points
    A,B=sample[:,:k],sample[:,k:]                       # Two pick-freeze base matrices
    YA,YB=model(A),model(B)                            # Evaluate initial samples
    variance=np.var(np.vstack([YA,YB]),axis=0,ddof=1) # Approximate output variances
    S1=np.full((k,len(OUTPUT_KEYS)),np.nan)            # Undefined for constant output
    ST=np.full_like(S1,np.nan)                         # Total-order variance contributions
    valid=variance>1e-24                               # Zero-variance outputs are undefined
    for j in range(k):
        AB=A.copy(); AB[:,j]=B[:,j]                     # Replace just parameter j
        YAB=model(AB)
        ST[j,valid]=np.mean((YA[:,valid]-YAB[:,valid])**2,axis=0)/(2*variance[valid])
        S1[j,valid]=1-np.mean((YB[:,valid]-YAB[:,valid])**2,axis=0)/(2*variance[valid])
    # Finite sampling may produce small negative S1 or ST slightly >1: leave un-clipped.
    return S1,ST,variance


In [41]:
# PART 5H — EXECUTE EXTERNAL SENSITIVITY FOR EACH INDIVIDUAL PCB/EMC GRADE
external_morris_rows,external_sobol_rows,external_nominal_rows=[],[],[]
EXTERNAL_PARAMS=["Coupled_T_RH", "Exposure_time"]  # Distinct independent model inputs
for grade_i, pair in enumerate(pairs):
    model=lambda u,p=pair: evaluate_external(u,p)   # Freeze identity of this particular grade
    mu,sig=morris(model,2,seed=SEED+grade_i)         # Morris screening of the two conditions
    S1,ST,var=sobol_jansen(model,2,seed=SEED+grade_i) # Sobol across environmental scenarios
    for j,param in enumerate(EXTERNAL_PARAMS):
        for z,out in enumerate(OUTPUT_KEYS):
            meta=dict(family=pair["family"],material=pair["material"],parameter=param,output=out)
            external_morris_rows.append({**meta,"mu_star":mu[j,z],"sigma":sig[j,z]})
            external_sobol_rows.append({**meta,"S1":S1[j,z],"ST":ST[j,z],"variance":var[z]})
    lo,hi=pair["low"],pair["high"]
    external_nominal_rows.append(dict(family=pair["family"],material=pair["material"],
            D_60=lo["D"],D_85=hi["D"],Csat_60=lo["Csat"],Csat_85=hi["Csat"],
            h_m=hi["h"],CME_beta=hi["beta"],CTE_ppmK=hi["alpha_lam"],
            E_MPa=hi["E"],nu_FIXED=hi["nu"],time_days_min=TIME_DAYS[0],
            time_days_max=TIME_DAYS[1],s_min=0,s_max=1))
external_morris=pd.DataFrame(external_morris_rows)
external_sobol=pd.DataFrame(external_sobol_rows)
external_nominal=pd.DataFrame(external_nominal_rows)
external_morris.to_csv(OUTDIR/"Morris_external_per_grade.csv",index=False)
external_sobol.to_csv(OUTDIR/"Sobol_external_per_grade.csv",index=False)
external_nominal.to_csv(OUTDIR/"External_input_endpoints.csv",index=False)
print(f"EXTERNAL: completed {len(pairs)} individual grades | {len(external_sobol)} Sobol result rows")


EXTERNAL: completed 21 individual grades | 168 Sobol result rows


In [42]:
# PART 5I — EXECUTE INTRINSIC SENSITIVITY AT EACH OF THE TWO SPECIFIC CONDITIONS
intrinsic_morris_rows,intrinsic_sobol_rows,intrinsic_range_rows=[],[],[]
for grade_i,pair in enumerate(pairs):
    for cond_i,condition in enumerate([LOW_CONDITION,HIGH_CONDITION]):
        if condition not in pair['selected_conditions']:  # Respect selected Excel preset row
            continue
        row=pair["low"] if cond_i==0 else pair["high"]   # Never mix condition-specific data
        model=lambda u,r=row: evaluate_intrinsic(u,r)  # Freeze grade + condition
        seed=SEED + grade_i*2+cond_i+1000
        mu,sig=morris(model,len(INTRINSIC_PARAMS),seed=seed)
        S1,ST,var=sobol_jansen(model,len(INTRINSIC_PARAMS),seed=seed)
        for j,param in enumerate(INTRINSIC_PARAMS):
            lo,hi=intrinsic_bounds(row)[param]
            intrinsic_range_rows.append(dict(family=pair["family"],material=pair["material"],
                condition=condition,parameter=param,nominal=float(row[param]),
                lower=lo,upper=hi,basis="ASSUMED screening bound, not measured uncertainty"))
            for z,out in enumerate(OUTPUT_KEYS):
                meta=dict(family=pair["family"],material=pair["material"],
                          condition=condition,parameter=param,output=out)
                intrinsic_morris_rows.append({**meta,"mu_star":mu[j,z],"sigma":sig[j,z]})
                intrinsic_sobol_rows.append({**meta,"S1":S1[j,z],"ST":ST[j,z],"variance":var[z]})
intrinsic_morris=pd.DataFrame(intrinsic_morris_rows)
intrinsic_sobol=pd.DataFrame(intrinsic_sobol_rows)
intrinsic_ranges=pd.DataFrame(intrinsic_range_rows)
intrinsic_morris.to_csv(OUTDIR/"Morris_intrinsic_per_condition.csv",index=False)
intrinsic_sobol.to_csv(OUTDIR/"Sobol_intrinsic_per_condition.csv",index=False)
intrinsic_ranges.to_csv(OUTDIR/"ASSUMED_intrinsic_ranges.csv",index=False)
print(f"INTRINSIC: {intrinsic_sobol[['material','condition']].drop_duplicates().shape[0]} material/condition runs")


INTRINSIC: 42 material/condition runs


In [43]:
# PART 5J — Keep machine-readable results; DO NOT combine grades in any chart.
ext_stress = (external_sobol.loc[external_sobol['output']=='Stress_Index_MPa']
    .pivot(index=['family','material'],columns='parameter',values='ST').reset_index())
ext_stress.columns.name = None
ext_stress = ext_stress.rename(columns={'Coupled_T_RH':'ST_joint_T_RH',
                                       'Exposure_time':'ST_time'})
ext_stress['higher_ST_factor'] = np.where(
    ext_stress['ST_joint_T_RH']>=ext_stress['ST_time'],
    'Joint T/RH', 'Exposure time')
ext_stress.to_csv(OUTDIR/'External_ST_stress_summary.csv',index=False)
print('Numerical results saved. NEXT: individual graph and table for every grade.')


Numerical results saved. NEXT: individual graph and table for every grade.


In [44]:
# PART 5K — Individual figure: 4 outputs × [MORRIS | SOBOL], one grade per figure
def safe_name(value):
    """Make one stable folder/filename from a supplier + grade string."""
    return re.sub(r'[^A-Za-z0-9_-]+','_',str(value)).strip('_')

def grade_directory(pair):
    """Never mix two materials in one results folder."""
    family_name = ('PCB' if pair['family']=='PCB laminate' else
                   'EMC' if pair['family']=='EMC encapsulation' else 'Package_Substrate')
    folder = OUTDIR/family_name/safe_name(pair['material'])
    folder.mkdir(parents=True,exist_ok=True)
    return folder

def draw_grade_report(pair, kind, condition=None):
    """Plot one material ONLY, showing all four outputs for one sensitivity study."""
    is_external = kind=='external'
    params = EXTERNAL_PARAMS if is_external else INTRINSIC_PARAMS
    m_source = external_morris if is_external else intrinsic_morris
    s_source = external_sobol if is_external else intrinsic_sobol
    # Filter to one material. Intrinsic data must also match one Excel condition.
    m = m_source.loc[m_source['material'].eq(pair['material'])]
    s = s_source.loc[s_source['material'].eq(pair['material'])]
    if not is_external:
        m = m.loc[m['condition'].eq(condition)]
        s = s.loc[s['condition'].eq(condition)]
    if m.empty or s.empty:
        raise ValueError(f'Missing per-grade data for {pair["material"]}: {condition}')

    fig,axes=plt.subplots(4,2,figsize=(11,12),squeeze=False)
    x = np.arange(len(params))
    labels=[PARAM_PRETTY[p] for p in params]
    for row_number, output in enumerate(OUTPUT_KEYS):
        # One model output occupies a separate ROW; Morris/Sobol never share an axis.
        mor = m.loc[m['output'].eq(output)].set_index('parameter').reindex(params)
        sob = s.loc[s['output'].eq(output)].set_index('parameter').reindex(params)
        if len(mor)!=len(params) or mor['mu_star'].isna().any():
            raise ValueError(f'Incomplete Morris results for {pair["material"]}: {output}')
        left,right=axes[row_number]
        left.bar(x,mor['mu_star'].to_numpy(),width=.65)
        left.set_ylabel('Morris μ* (output units)')
        left.set_title(OUTPUT_PRETTY[output]+' — Morris')
        left.grid(axis='y',alpha=.2)
        left.tick_params(axis='y',labelsize=9)
        # S1/ST use the same variance basis but can exhibit sampling noise.
        width=.37
        right.bar(x-width/2,sob['S1'].to_numpy(),width,label='S1')
        right.bar(x+width/2,sob['ST'].to_numpy(),width,label='ST')
        right.axhline(0,color='black',linewidth=.6)
        right.set_ylabel('Sobol variance fraction')
        right.set_title(OUTPUT_PRETTY[output]+' — Sobol')
        right.grid(axis='y',alpha=.2)
        right.legend(fontsize=9,loc='upper right')
        for ax in (left,right):
            ax.set_xticks(x)
            ax.set_xticklabels(labels,rotation=25,ha='right',fontsize=9)
    label = 'EXTERNAL · joint T/RH + exposure time' if is_external else f'INTRINSIC · {condition}'
    fig.suptitle(pair['material']+' | '+label,fontsize=17,weight='bold',y=.999)
    note = (f"Sources: 60/60={pair['low']['preset_source']}, 85/85={pair['high']['preset_source']}; time 0.25–7 days."
            if is_external else
            f"Source: {(pair['low'] if condition==LOW_CONDITION else pair['high'])['preset_source']}; D ×/÷2; others ±10%; ν fixed.")
    fig.text(.5,.007,note,ha='center',fontsize=9)
    fig.subplots_adjust(left=.10,right=.97,top=.955,bottom=.065,hspace=.59,wspace=.34)
    folder=grade_directory(pair)
    name='01_EXTERNAL_Morris_Sobol.png' if is_external else (
        '02_INTRINSIC_'+('60C_60RH' if condition==LOW_CONDITION else '85C_85RH')+'_Morris_Sobol.png')
    path=folder/name
    fig.savefig(path,dpi=115)
    return fig,path

# Save merged Streamlit/Excel presets + sensitivity CSV tables to EACH grade's folder.
for pair in pairs:
    folder=grade_directory(pair)
    inputs=[]
    for condition,row in [(LOW_CONDITION,pair['low']),(HIGH_CONDITION,pair['high'])]:
        inputs.append({'family':pair['family'],'material':pair['material'],
            'condition':condition,'Excel_row':row['excel_row'], 'preset_source':row['preset_source'],
            'preset_name':row['preset_name'],
            **{key:float(row[key]) for key in INPUT_NAMES}})
    pd.DataFrame(inputs).to_csv(folder/'00_Selected_source_presets.csv',index=False)
    external_morris.loc[external_morris['material'].eq(pair['material'])].to_csv(
        folder/'01_external_Morris_table.csv',index=False)
    external_sobol.loc[external_sobol['material'].eq(pair['material'])].to_csv(
        folder/'01_external_Sobol_S1_ST_table.csv',index=False)
    for condition in [LOW_CONDITION,HIGH_CONDITION]:
        short='60C_60RH' if condition==LOW_CONDITION else '85C_85RH'
        for frame,filename in [(intrinsic_morris,'Morris'),(intrinsic_sobol,'Sobol_S1_ST'),
                               (intrinsic_ranges,'assumed_input_ranges')]:
            frame.loc[frame['material'].eq(pair['material']) &
                      frame['condition'].eq(condition)].to_csv(
                folder/f'{short}_{filename}.csv',index=False)

figure_index=[]
shown=0
for pair in pairs:
    fig,path=draw_grade_report(pair,'external')  # One graph ONLY for this material
    figure_index.append({'family':pair['family'],'material':pair['material'],
                         'study':'External','condition':'60/60 → 85/85',
                         'file':str(path.relative_to(OUTDIR))})
    if shown<PLOT_INLINE_LIMIT:
        plt.show();shown+=1
    plt.close(fig)
print(f'Created {len(pairs)} separate EXTERNAL charts (one per grade).')


Created 21 separate EXTERNAL charts (one per grade).


In [45]:
# PART 5L — Separate INTRINSIC charts for each grade at each selected Excel condition
intrinsic_chart_count=0
for pair in pairs:
    for condition in [LOW_CONDITION,HIGH_CONDITION]:
        if condition not in pair['selected_conditions']:
            continue
        fig,path=draw_grade_report(pair,'intrinsic',condition=condition)
        figure_index.append({'family':pair['family'],'material':pair['material'],
                             'study':'Intrinsic','condition':condition,
                             'file':str(path.relative_to(OUTDIR))})
        if shown<PLOT_INLINE_LIMIT:
            plt.show();shown+=1
        plt.close(fig)
        intrinsic_chart_count+=1
pd.DataFrame(figure_index).to_csv(OUTDIR/'00_Find_Individual_Charts.csv',index=False)
print(f'Created {intrinsic_chart_count} separate INTRINSIC charts.')
print(f'All charts and grade-specific result tables saved under: {OUTDIR.resolve()}')
display(pd.DataFrame(figure_index).head(12))


Created 42 separate INTRINSIC charts.
All charts and grade-specific result tables saved under: C:\Users\yong jiang\Desktop\Y4S1\3015\Updated 111026\Dual_source_individual_reports


,family,material,study,condition,file
0,EMC encapsulation,Nitto Denko GE-100,External,60/60 → 85/85,EMC\Nitto_Denko_GE-100\01_EXTERNAL_Morris_Sobo...
1,EMC encapsulation,Panasonic CV-8300,External,60/60 → 85/85,EMC\Panasonic_CV-8300\01_EXTERNAL_Morris_Sobol...
2,EMC encapsulation,Panasonic CV8200,External,60/60 → 85/85,EMC\Panasonic_CV8200\01_EXTERNAL_Morris_Sobol.png
3,EMC encapsulation,Panasonic CV8700,External,60/60 → 85/85,EMC\Panasonic_CV8700\01_EXTERNAL_Morris_Sobol.png
4,EMC encapsulation,Panasonic CV8710U,External,60/60 → 85/85,EMC\Panasonic_CV8710U\01_EXTERNAL_Morris_Sobol...
5,EMC encapsulation,Resonac CEL-9200,External,60/60 → 85/85,EMC\Resonac_CEL-9200\01_EXTERNAL_Morris_Sobol.png
6,EMC encapsulation,Resonac CEL-9750,External,60/60 → 85/85,EMC\Resonac_CEL-9750\01_EXTERNAL_Morris_Sobol.png
7,EMC encapsulation,Sumitomo Bakelite EME-A700,External,60/60 → 85/85,EMC\Sumitomo_Bakelite_EME-A700\01_EXTERNAL_Mor...
8,EMC encapsulation,Sumitomo Bakelite EME-E500,External,60/60 → 85/85,EMC\Sumitomo_Bakelite_EME-E500\01_EXTERNAL_Mor...
9,EMC encapsulation,Sumitomo Bakelite EME-G700,External,60/60 → 85/85,EMC\Sumitomo_Bakelite_EME-G700\01_EXTERNAL_Mor...


In [46]:
# PART 5M — Convergence check for up to three grades in the CHOSEN run
# This also works when SENSITIVITY_SCOPE='selected' and only one material is chosen.
convergence=[]
for pair in pairs[:3]:
    model=lambda u,p=pair:evaluate_external(u,p)
    for n in [512,1024,2048]:
        S1,ST,var=sobol_jansen(model,2,n=n,seed=SEED+777)
        for j,param in enumerate(EXTERNAL_PARAMS):
            z=OUTPUT_KEYS.index('Stress_Index_MPa')
            convergence.append({'material':pair['material'],'N':n,'parameter':param,
                'output':'Stress_Index_MPa','S1':S1[j,z],'ST':ST[j,z]})
convergence_df=pd.DataFrame(convergence)
convergence_df.to_csv(OUTDIR/'Sobol_convergence_check.csv',index=False)
print('Representative convergence check:')
display(convergence_df.pivot_table(index=['material','parameter'],columns='N',values='ST').round(3))


Representative convergence check:


N                                  512    1024   2048
material           parameter                         
Nitto Denko GE-100 Coupled_T_RH   0.119  0.119  0.119
                   Exposure_time  0.952  0.951  0.949
Panasonic CV-8300  Coupled_T_RH   0.222  0.222  0.222
                   Exposure_time  0.841  0.840  0.838
Panasonic CV8200   Coupled_T_RH   0.102  0.102  0.102
                   Exposure_time  0.969  0.968  0.967

## PART 5N — Understanding the individual reports

**Each grade has its own folder** and its own figures; no figure combines two materials. Open `00_Find_Individual_Charts.csv` to locate each PNG. Inside a grade folder, `00_Selected_source_presets.csv` records both source Excel rows and the exact properties used. External sensitivity uses the grade's two environmental endpoints; intrinsic sensitivity uses only the grade's selected 60/60 and/or 85/85 condition(s).

**Morris μ\***: mean absolute elementary effect in the units of the corresponding output. **Morris σ** is saved in the CSV tables, quantifying variation among elementary effects (possible interactions or nonlinearity). **Sobol S1** is the first-order contribution to the output variance; **ST** counts the input's contribution including interactions. Small negative S1 values can arise from sampling noise and should not be interpreted as a physically negative effect.

**Fixed in the intrinsic study**: each grade's Excel exposure time, thickness, copper CTE, temperature difference, and Poisson's ratio. The source Excel's 60°C/60%RH and 85°C/85%RH values are kept separate; D and Csat are NOT taken from another material. **Changed in the intrinsic study**: D ×/÷2 (log-uniform), Csat/CME/CTE/E ±10% (uniform). These are sensitivity-screening assumptions, not validated measurement tolerances.

**External limitation**: temperature and humidity vary together because this workbook only supplies 60/60 and 85/85 endpoints. The interpolation assumptions are modelled scenarios, not independent thermal and humidity regressions. External Sobol indices and intrinsic Sobol indices must not be combined as if their input uncertainties were the same. The constrained-stress index does NOT give actual Cu-pad edge, corner, or interfacial peeling stress.

### Example: select one grade
Set `SELECTED = [0, 1]` in Part 3 for the two Panasonic CV8710U conditions; set `SENSITIVITY_SCOPE = 'selected'` in Part 5A, then run the notebook top-to-bottom. For **all grades in separate folders**, set `SENSITIVITY_SCOPE = 'all'`.


In [47]:
# PART 5O — ZIP all SEPARATE grade reports for convenient download
from zipfile import ZipFile, ZIP_DEFLATED
# Include source reconciliation so property provenance remains verifiable.
preset_audit.to_csv(OUTDIR/'00_Material_source_audit.csv',index=False)

zip_path=OUTDIR.with_suffix('.zip')
all_files=sorted(path for path in OUTDIR.rglob('*') if path.is_file())
with ZipFile(zip_path,'w',compression=ZIP_DEFLATED,compresslevel=6) as zf:
    for path in all_files:
        zf.write(path,arcname=path.relative_to(OUTDIR))
print(f'ZIP ready: {zip_path} ({len(all_files)} individual files)')
print('No material plots were combined. Each folder contains its own inputs, charts, tables.')


ZIP ready: Dual_source_individual_reports.zip (262 individual files)
No material plots were combined. Each folder contains its own inputs, charts, tables.


# PART 6 — STANDALONE MORRIS ANALYSIS (ONE MATERIAL PER FIGURE)

The earlier **Part 5F** already runs Morris using one-at-a-time trajectories (40 trajectories, 6 discrete grid levels, step 0.6 in normalized input space). **Part 6 makes MORRIS-ONLY plots** and ranked tables from those computed elementary effects, independently of the Sobol figures. No values from one grade are mixed with another grade.

- **External Morris:** one plot per material using **joint temperature/RH severity** (60°C/60%RH to 85°C/85%RH) and **log-uniform exposure time** (0.25–7 days). Because they move together, the analysis does *not* separate temperature from RH.
- **Intrinsic Morris:** separate plots for the same material at **60°C/60%RH and 85°C/85%RH** using that **selected preset**'s properties. Five inputs vary: **D** (0.5×–2×; log-uniform), and **Csat, CME β, CTE α, Eeff** (each ±10%; uniform). The preset's ν, copper CTE, thickness, ΔT and 24-hour exposure remain fixed.
- **Morris μ\***: mean of the *absolute* elementary effects (overall screening importance). **Morris σ**: standard deviation of signed elementary effects (variation due to nonlinearity and/or input interactions). Each output has its own physical units; comparing μ* numerically across different outputs is invalid.
- Each plot has 4 panels: fractional uptake, absorbed water density, hygroscopic strain, and constrained stress **index**. These are model results, not measured uncertainty bounds or interfacial pad stresses.

Run Parts 1–5 first. If you set `SENSITIVITY_SCOPE = 'selected'` and specify one Streamlit preset in **Part 5A**, Part 6 will only plot that selection. Otherwise, it generates individual reports for every material in your merged Excel + Streamlit library.

In [48]:
# PART 6A — Create a MORRIS-ONLY scatter-free bar figure and a ranking table per preset.
# This uses Morris results calculated earlier in Parts 5H and 5I, not Sobol indices.
from zipfile import ZipFile, ZIP_DEFLATED

MORRIS_DIR = Path('Morris_only_individual_reports')
MORRIS_DIR.mkdir(parents=True, exist_ok=True)
MORRIS_BAR_DPI = 150

def morris_folder(pair):
    """One output directory per material, including the family to prevent collisions."""
    family = ('PCB' if pair['family'] == 'PCB laminate' else
              'EMC' if pair['family'] == 'EMC encapsulation' else 'Package_Substrate')
    folder = MORRIS_DIR / family / safe_name(pair['material'])
    folder.mkdir(parents=True, exist_ok=True)
    return folder

def plot_morris_only(pair, kind, condition=None):
    """Plot μ* and σ for FOUR outputs using ONE material only."""
    is_external = (kind == 'external')
    params = EXTERNAL_PARAMS if is_external else INTRINSIC_PARAMS
    table = external_morris if is_external else intrinsic_morris
    rows = table.loc[(table['family'] == pair['family']) &
                     (table['material'] == pair['material'])].copy()
    if not is_external:
        rows = rows.loc[rows['condition'] == condition]
    if rows.empty:
        raise ValueError('No Morris results for ' + pair['material'] + ' / ' + str(condition))

    folder = morris_folder(pair)
    tag = ('EXTERNAL' if is_external else
           ('INTRINSIC_60C_60RH' if condition == LOW_CONDITION else 'INTRINSIC_85C_85RH'))

    # Save a detailed table containing ALL FOUR outputs and the ranking within each output.
    rows['rank_by_mu_star'] = (rows.groupby('output')['mu_star']
                                 .rank(ascending=False, method='min').astype(int))
    rows.sort_values(['output', 'rank_by_mu_star']).to_csv(
        folder / f'{tag}_Morris_rankings.csv', index=False)

    fig, axes = plt.subplots(2, 2, figsize=(12.4, 8.6))
    names = [PARAM_PRETTY.get(k, k) for k in params]
    y_pos = np.arange(len(params))
    for ax, output in zip(axes.ravel(), OUTPUT_KEYS):
        these = rows.loc[rows['output'] == output].set_index('parameter').reindex(params)
        if these[['mu_star', 'sigma']].isna().any().any():
            raise ValueError('Missing Morris estimate: ' + pair['material'] + ' / ' + output)
        # μ* and σ have the SAME units for any given output (never mix different outputs).
        ax.barh(y_pos - 0.18, these['mu_star'].to_numpy(), height=0.34,
                label='μ* — mean absolute effect')
        ax.barh(y_pos + 0.18, these['sigma'].to_numpy(), height=0.34,
                label='σ — variation in effect')
        ax.set_yticks(y_pos, names)
        ax.invert_yaxis()
        ax.set_title(OUTPUT_PRETTY[output], fontsize=11, weight='bold')
        ax.set_xlabel('Morris effect (units of this output)', fontsize=9)
        ax.grid(axis='x', alpha=0.20)
        ax.set_axisbelow(True)
        ax.tick_params(axis='x', labelsize=8)
        ax.ticklabel_format(axis='x', style='sci', scilimits=(-3, 4))
    axes[0, 0].legend(fontsize=8, loc='lower right')

    title = ('Environmental Morris — joint T/RH and time' if is_external else
             f'Intrinsic Morris — {condition}')
    fig.suptitle(pair['material'] + ' | ' + title,
                 fontsize=14, fontweight='bold', y=.99)
    provenance = ('60/60: ' + str(pair['low']['preset_source']) +
                  ' | 85/85: ' + str(pair['high']['preset_source']))
    setting = ('Joint 60/60 → 85/85; time 0.25–7 days' if is_external else
               'D ×0.5–2; Csat/CME/CTE/E ±10%; ν fixed; time = preset value')
    fig.text(.5, .017, provenance + ' | ' + setting, ha='center', fontsize=8)
    fig.tight_layout(rect=[0, .045, 1, .94], w_pad=2.5, h_pad=2.1)
    plot_path = folder / f'{tag}_Morris_only.png'
    fig.savefig(plot_path, dpi=MORRIS_BAR_DPI, bbox_inches='tight')
    plt.close(fig)  # Important: do not retain 63 Matplotlib figures in Jupyter memory.
    return plot_path, folder / f'{tag}_Morris_rankings.csv'

In [49]:
# PART 6B — Generate separated plots and tables using the merged source preset selection.
morris_file_index = []
for pair in pairs:
    # Keep a copy of the exact input presets (two environmental rows per grade).
    folder = morris_folder(pair)
    input_rows = []
    for cond, row in [(LOW_CONDITION, pair['low']), (HIGH_CONDITION, pair['high'])]:
        input_rows.append({'material': pair['material'], 'condition': cond,
            'preset_source': row['preset_source'], 'preset_name': row['preset_name'],
            **{key: float(row[key]) for key in INPUT_NAMES}})
    pd.DataFrame(input_rows).to_csv(folder / '00_Input_presets_used.csv', index=False)

    # External Morris: one graph for this material, using its own condition endpoints.
    fig_path, table_path = plot_morris_only(pair, 'external')
    morris_file_index.append({'material': pair['material'], 'family': pair['family'],
        'study': 'external', 'condition': 'coupled 60/60–85/85',
        'graph': str(fig_path.relative_to(MORRIS_DIR)),
        'table': str(table_path.relative_to(MORRIS_DIR))})

    # Intrinsic Morris: independent graph for each requested condition.
    for condition in [LOW_CONDITION, HIGH_CONDITION]:
        if condition not in pair['selected_conditions']:
            continue
        fig_path, table_path = plot_morris_only(pair, 'intrinsic', condition)
        morris_file_index.append({'material': pair['material'], 'family': pair['family'],
            'study': 'intrinsic', 'condition': condition,
            'graph': str(fig_path.relative_to(MORRIS_DIR)),
            'table': str(table_path.relative_to(MORRIS_DIR))})

morris_index = pd.DataFrame(morris_file_index)
morris_index.to_csv(MORRIS_DIR / '00_Morris_chart_and_table_index.csv', index=False)
external_morris.to_csv(MORRIS_DIR / '00_Morris_external_all_results.csv', index=False)
intrinsic_morris.to_csv(MORRIS_DIR / '00_Morris_intrinsic_all_results.csv', index=False)
intrinsic_ranges.to_csv(MORRIS_DIR / '00_ASSUMED_intrinsic_ranges.csv', index=False)
print(f'Morris-only analysis: {len(pairs)} individual materials and {len(morris_index)} separate charts')
display(morris_index.head(6))

Morris-only analysis: 21 individual materials and 63 separate charts


,material,family,study,condition,graph,table
0,Nitto Denko GE-100,EMC encapsulation,external,coupled 60/60–85/85,EMC\Nitto_Denko_GE-100\EXTERNAL_Morris_only.png,EMC\Nitto_Denko_GE-100\EXTERNAL_Morris_ranking...
1,Nitto Denko GE-100,EMC encapsulation,intrinsic,60°C/60%RH,EMC\Nitto_Denko_GE-100\INTRINSIC_60C_60RH_Morr...,EMC\Nitto_Denko_GE-100\INTRINSIC_60C_60RH_Morr...
2,Nitto Denko GE-100,EMC encapsulation,intrinsic,85°C/85%RH,EMC\Nitto_Denko_GE-100\INTRINSIC_85C_85RH_Morr...,EMC\Nitto_Denko_GE-100\INTRINSIC_85C_85RH_Morr...
3,Panasonic CV-8300,EMC encapsulation,external,coupled 60/60–85/85,EMC\Panasonic_CV-8300\EXTERNAL_Morris_only.png,EMC\Panasonic_CV-8300\EXTERNAL_Morris_rankings...
4,Panasonic CV-8300,EMC encapsulation,intrinsic,60°C/60%RH,EMC\Panasonic_CV-8300\INTRINSIC_60C_60RH_Morri...,EMC\Panasonic_CV-8300\INTRINSIC_60C_60RH_Morri...
5,Panasonic CV-8300,EMC encapsulation,intrinsic,85°C/85%RH,EMC\Panasonic_CV-8300\INTRINSIC_85C_85RH_Morri...,EMC\Panasonic_CV-8300\INTRINSIC_85C_85RH_Morri...


In [50]:
# PART 6C — ZIP all independent Morris graphs + result tables for download.
morris_zip = MORRIS_DIR.with_suffix('.zip')
with ZipFile(morris_zip, 'w', compression=ZIP_DEFLATED, compresslevel=6) as z:
    for file in sorted(MORRIS_DIR.rglob('*')):
        if file.is_file():
            z.write(file, arcname=file.relative_to(MORRIS_DIR))
print('Download bundle:', morris_zip.resolve())

Download bundle: C:\Users\yong jiang\Desktop\Y4S1\3015\Updated 111026\Morris_only_individual_reports.zip
